# Orçamento de otimização dos Transformers — calibração e execução completa (2 T4)

**O que se mede.** No protocolo publicado os seis Transformers recebem `max_epochs=40, patience=6`
(`src/tuning/grids.py`), constantes do estudo inteiro: a grade varre só arquitetura
(`num_blocks`/`num_heads`/`n_layers`/`m_ratio`), nunca o orçamento, e `lr` é fixa em 1e-3. Os LSSVMs,
em contraste, param por **tolerância de convergência** (`tol=1e-6`, teto 500), e esse teto está
verificado como não-restritivo: em `results/admm_stability_knobs.json` o braço "teto 500" trunca o
CREDIT em 500 iterações contra 673–812 do livre e o F1 é idêntico até a quarta casa.

**Quem está parando os Transformers não é o teto, é a paciência.** Nos `n_epochs` do piloto sob o
protocolo publicado, FT-softmax para no TWS em 10/9/8 épocas e no HAB em 10/7/11; FT-CUR no TWS em
18/10/10. Como no Tier 1 o lote (512) é maior que o conjunto de ajuste, **uma época é um passo de
gradiente** — o critério efetivo é "pare após 6 passos sem melhora de `val_loss`", medido numa
validação de poucas dezenas de amostras dentro das dobras do `GridSearchCV`.

**Por que a grade é re-selecionada dentro de cada braço.** Congelar os hiperparâmetros escolhidos sob
o orçamento apertado enviesaria o efeito para baixo, porque a seleção favorece o que treina rápido: o
SAINT escolhe `n_layers=1` em 53% (Tier 1) e 63% (Tier 2), contra 33% do uniforme. Nas variantes FT a
seleção é praticamente uniforme (16–20% nas seis células), ou seja, em 40 épocas o escore de CV **não
distingue arquitetura** — a moda dessa distribuição é moda de ruído. No teste de sanidade local a
arquitetura escolhida mudou ao trocar o orçamento (FT-softmax 3 blocos/4 cabeças → 2/2; SAINT
2 cabeças → 4). Portanto compara-se **protocolo** (grade + orçamento), não parâmetros entre orçamentos.

**Instrumentação nova.** Os laços de treino passaram a gravar `best_epoch` (época do checkpoint
restaurado), `n_steps`, `steps_per_epoch` e `stopped_early`. `best_epoch` é a medida que decide a
questão: se ele se acumula bem abaixo do teto, o orçamento não aperta; se encosta nele, os números
publicados são limitados por orçamento.

**Ordem de execução.** Rode as células 1 a 5 (**calibração**, ≈1h30 de sessão) e me passe os quatro
JSONs: a célula 6 imprime a razão de custo medida e extrapola a execução completa. Só então rode as
células 7 e 8. A calibração também serve de controle de reprodutibilidade: o braço publicado de 2
sementes tem de bater com `results/tier1_gridcv.json`.

**Antes de rodar:** Settings → Accelerator → **GPU T4 x2**. Tudo resumível; parciais espelhados em
`/kaggle/working` a cada atualização de progresso.

In [ ]:
# ── 1. GPUs ──
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2 (com 1 placa, troque run_phase_2gpu por run_phase)'

In [ ]:
# ── 2. Repositório + verificação da instrumentação de orçamento ──
import os, subprocess, json, statistics as st
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
!git log --oneline -2
import sys; sys.path.insert(0, '.')

# A instrumentação tem de existir nos DOIS laços: o FT tem laço próprio, enquanto
# SAINT e FT-CUR passam por fit_model/train_epoch em ft_transformer_model.py.
import inspect
from src.models.transformers.ft_transformer import FTTransformer
assert 'self.best_epoch_' in inspect.getsource(FTTransformer.fit), 'FT sem best_epoch_ — clone velho, refaça o pull'
assert "'best_epoch'" in open('src/models/ft_transformer_model.py').read(), 'fit_model sem best_epoch'

import scripts.run_tier1_gridcv as _r1
assert hasattr(_r1, 'BUDGET_OVERRIDE') and hasattr(_r1, '_apply_budget'), 'runner sem override de orçamento'
# O override precisa acertar a chave de cada wrapper (FT usa max_epochs; SAINT/FT-CUR usam epochs).
_r1.BUDGET_OVERRIDE.update({'epochs': 200, 'patience': 16})
assert _r1._apply_budget({'max_epochs': 40, 'patience': 6}) == {'max_epochs': 200, 'patience': 16}
assert _r1._apply_budget({'epochs': 40, 'patience': 6})     == {'epochs': 200, 'patience': 16}
_r1.BUDGET_OVERRIDE.clear()
print('instrumentação OK: best_epoch/n_steps nos dois laços + override nas duas convenções de chave')

In [ ]:
# ── 3. Dependências e dados (tudo em cache no clone; nada é gerado nem baixado) ──
!pip install -q einops scikit-posthocs openpyxl 2>&1 | tail -n 1
from pathlib import Path
USADOS = ['BCW','PID','HAB','VCP','GCR','AUS','AI4I','TWS','TWM','TWC',        # Tier 1
          'ADULT','BANK','CREDIT','HIGGS50K','SHOPPERS','TELCO']               # Tier 2
falt = [d for d in USADOS if not Path(f'data/raw/{d.upper()}.parquet').exists()]
assert not falt, f'sem cache (seriam gerados em paralelo): {falt} — refaça o pull'
from src.data.loaders import DatasetLoader
for ds in USADOS:
    X, y, _ = DatasetLoader.load(ds)
print(f'{len(USADOS)} datasets OK — todos em cache, nenhuma escrita concorrente possível')

In [ ]:
# ── 4. Configuração, progresso e paralelismo nas duas placas ──
import json, os, shutil, time, subprocess, collections, statistics as st
from pathlib import Path

MODELS = ['FTTransformer_softmax', 'FTTransformer_topk', 'FTTransformer_entmax',
          'FTTransformer_sparsemax', 'SAINTColnorm', 'FTTransformerCURColnorm']
MODELS_STR = ' '.join(MODELS)
TIER1 = 'BCW PID HAB VCP GCR AUS AI4I TWS TWM TWC'
TIER2 = 'ADULT BANK CREDIT HIGGS50K SHOPPERS TELCO'
# Calibração do Tier 2: os três que mais custam e mais variam em passos por época.
TIER2_CALIB = 'BANK HIGGS50K TELCO'

# Orçamento do braço novo. min_epochs fica DESLIGADO de propósito: forçar um piso
# destruiria a medição de onde a paciência dispara — foi o que inviabilizou o piloto
# de 18/09, que usava min_epochs=200 e por isso não serve para estimar custo.
BUDGET = '--budget-epochs 200 --budget-patience 16'

SEEDS_CALIB = range(2)      # 1 semente por placa
SEEDS_FULL  = range(10)     # 5 por placa

OUT = {'c1pub': 'results/budget_calib_t1_pub.json',  'c1new': 'results/budget_calib_t1_200.json',
       'c2pub': 'results/budget_calib_t2_pub.json',  'c2new': 'results/budget_calib_t2_200.json',
       'f1new': 'results/budget_full_t1_200.json',   'f2new': 'results/budget_full_t2_200.json'}
nm, nc, nf = len(MODELS), len(list(SEEDS_CALIB)), len(list(SEEDS_FULL))
TOTAL = {'c1pub': nm*10*nc, 'c1new': nm*10*nc,
         'c2pub': nm*3*nc,  'c2new': nm*3*nc,
         'f1new': nm*10*nf, 'f2new': nm*6*nf}
Path('results').mkdir(exist_ok=True)

RESUME_DIR = None    # ex.: Path('/kaggle/input/budget-parcial') para retomar sessão anterior
if RESUME_DIR:
    for k, v in OUT.items():
        src = Path(RESUME_DIR) / Path(v).name
        if src.exists(): shutil.copy(src, v); print('restaurado', v)

def save(key):
    shutil.copy(OUT[key], '/kaggle/working/' + Path(OUT[key]).name)
    print('salvo em Output:', Path(OUT[key]).name)

def _mirror(*paths):
    # Os runners gravam em <repo>/results/ a cada execução (append atômico), mas o painel
    # Output do Kaggle mostra /kaggle/working: sem esta cópia periódica, uma sessão
    # interrompida deixaria os dados só no diretório aninhado do clone.
    for p in paths:
        try:
            if Path(p).exists(): shutil.copy(p, '/kaggle/working/' + Path(p).name)
        except Exception as e:
            print('  (aviso: falha ao espelhar', p, e, ')')

def _count(path):
    # Só erros ESPERADOS são tolerados (arquivo ainda não criado / meio de escrita).
    # Um except amplo aqui esconderia um bug de import e o contador ficaria em 0 para
    # sempre com os processos rodando — foi o que aconteceu em 2026-09-19 no SAINT.
    try: r = json.load(open(path))
    except (FileNotFoundError, ValueError): return 0, ''
    ok = [x for x in r if x.get('status', 'ok') == 'ok']
    if not ok: return 0, ''
    x = ok[-1]; f1 = x.get('test_f1_macro')
    info = 'último: ' + str(x.get('dataset', '?')) + '/seed' + str(x.get('seed', '?'))
    if isinstance(f1, (int, float)): info += f' F1={f1:.3f}'
    if x.get('best_epoch') is not None:
        info += f" best_ep={x['best_epoch']}/{x.get('n_epochs','?')}"
    return len(ok), info

def run_phase_2gpu(key, cmd_fmt, seeds, every=60, heartbeat=900):
    # Duas T4: um processo por placa, metade das sementes cada, shards separados.
    # NÃO usar DataParallel/DDP: FT-CUR e SAINT fazem atenção inter-instâncias DENTRO
    # do lote, então dividir o lote entre placas mudaria o modelo. O paralelismo aqui é
    # por semente, que não altera nada. Dois processos no mesmo JSON se sobrescreveriam,
    # daí um arquivo por placa.
    seeds = list(seeds); half = max(1, len(seeds)//2)
    base, total = OUT[key], TOTAL[key]
    shards, procs, logs = [], [], []
    for gpu, sds in [(0, seeds[:half]), (1, seeds[half:])]:
        if not sds: continue
        shard = base.replace('.json', f'_g{gpu}.json'); shards.append(shard)
        log = f'/kaggle/working/{key}_g{gpu}.log'; logs.append(log)
        cmd = cmd_fmt.format(seeds=' '.join(map(str, sds)), output=shard)
        lf = open(log, 'w')
        # CUDA_VISIBLE_DEVICES prende o processo a uma placa; os limites de thread evitam
        # que os dois disputem os poucos vCPUs do Kaggle (oversubscription de BLAS/OMP
        # chega a dobrar o tempo do pré-processamento).
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
                   OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2',
                   NUMEXPR_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false')
        procs.append(subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT, env=env))
        print(f'[{key}] GPU {gpu}: sementes {sds[0]}..{sds[-1]} -> {Path(shard).name}', flush=True)
    t0 = time.time(); last_n, last_print = -1, 0.0
    while True:
        rcs = [p.poll() for p in procs]
        n = sum(_count(s)[0] for s in shards); now = time.time()
        if n != last_n or all(rc is not None for rc in rcs) or now - last_print > heartbeat:
            el = (now - t0)/60; eta = el/n*(total-n) if n else float('nan')
            _, info = _count(shards[0])
            print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min  {info}', flush=True)
            _mirror(*shards)
            last_n, last_print = n, now
        if all(rc is not None for rc in rcs): break
        time.sleep(every)
    recs = []
    for s in shards:
        try: recs += json.load(open(s))
        except Exception: pass
    Path(base).write_text(json.dumps(recs, indent=1))
    print(f'[{key}] terminou (exits={rcs}) — {len(recs)} registros'); save(key)
    for s in shards: shutil.copy(s, '/kaggle/working/' + Path(s).name)
    if any(rc != 0 for rc in rcs): raise RuntimeError(f'{key}: exits {rcs} — ver {logs}')

for _n in ('json', 'os', 'shutil', 'time', 'subprocess', 'collections', 'st', 'Path'):
    assert _n in dir() or _n in globals(), f'falta import: {_n}'
print('config OK —', len(MODELS), 'Transformers | orçamento novo:', BUDGET)
for k, v in OUT.items(): print(f'  {k:<6} -> {v}  (total {TOTAL[k]})')

In [ ]:
# ── 5. CALIBRAÇÃO (≈1h30) — quatro braços de 2 sementes ──
# Tier 1 nos dois orçamentos. O braço publicado também é controle de reprodutibilidade:
# tem de bater com results/tier1_gridcv.json nas sementes 0–1.
run_phase_2gpu('c1pub', "python -u scripts/run_tier1_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER1 + " --seeds {seeds} --output {output} --log-level WARNING",
               seeds=SEEDS_CALIB)
run_phase_2gpu('c1new', "python -u scripts/run_tier1_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER1 + " " + BUDGET +
               " --seeds {seeds} --output {output} --log-level WARNING",
               seeds=SEEDS_CALIB)
# Tier 2 nos três datasets mais caros: é aqui que o teto de 200 pode encostar, porque
# N=2000 com lote 512 dá vários passos por época, enquanto o Tier 1 dá um só.
run_phase_2gpu('c2pub', "python -u scripts/run_tier2_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER2_CALIB + " --n-train 2000" +
               " --seeds {seeds} --output {output} --log-level WARNING",
               seeds=SEEDS_CALIB)
run_phase_2gpu('c2new', "python -u scripts/run_tier2_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER2_CALIB + " --n-train 2000 " + BUDGET +
               " --seeds {seeds} --output {output} --log-level WARNING",
               seeds=SEEDS_CALIB)
print('\nCalibração concluída. Baixe budget_calib_*.json de /kaggle/working e rode a célula 6.')

In [ ]:
# ── 6. Leitura da calibração: razão de custo, onde a paciência dispara, extrapolação ──
def _load(k):
    p = Path(OUT[k])
    return [r for r in json.load(open(p)) if r.get('status') == 'ok'] if p.exists() else []

def _resumo(pub, new, rotulo, n_ds_full, n_seeds_full):
    if not pub or not new:
        print(f'{rotulo}: faltam dados ({len(pub)} pub, {len(new)} novo)'); return None
    tp, tn = sum(r['fit_time_s'] for r in pub), sum(r['fit_time_s'] for r in new)
    print(f'\n===== {rotulo} =====')
    print(f'custo: publicado {tp/60:.1f} min | orçamento 200/16 {tn/60:.1f} min | razão {tn/tp:.2f}x')
    cab = ('modelo', 'braço', 'best_ep', 'n_ep', 'passos', 'no teto', 'F1')
    print(f'\n{cab[0]:<26} {cab[1]:<9} {cab[2]:>8} {cab[3]:>6} {cab[4]:>7} {cab[5]:>9} {cab[6]:>7}')
    for mdl in MODELS:
        for tag, rs in (('publicado', pub), ('200/16', new)):
            sel = [r for r in rs if r['variant'] == mdl and r.get('best_epoch') is not None]
            if not sel: continue
            # "no teto" = não parou por paciência, ou seja bateu no máximo de épocas.
            teto = sum(1 for r in sel if not r.get('stopped_early', False))
            print(f"{mdl:<26} {tag:<9} {st.median([r['best_epoch'] for r in sel]):8.0f} "
                  f"{st.median([r['n_epochs'] for r in sel]):6.0f} "
                  f"{st.median([r['n_steps'] for r in sel]):7.0f} "
                  f"{teto:>5}/{len(sel):<3} {st.mean([r['test_f1_macro'] for r in sel]):7.4f}")
    # Extrapolação: custo médio por execução do braço novo × grade completa do tier.
    por_run = tn / len(new)
    horas = por_run * len(MODELS) * n_ds_full * n_seeds_full / 3600
    print(f'\nextrapolação do braço 200/16 completo ({n_ds_full} datasets × {n_seeds_full} sementes '
          f'× {len(MODELS)} modelos): {horas:.1f} h de ajuste ≈ {horas/2:.1f} h de sessão em 2 placas')
    return horas

h1 = _resumo(_load('c1pub'), _load('c1new'), 'TIER 1 (10 datasets)', 10, len(list(SEEDS_FULL)))
h2 = _resumo(_load('c2pub'), _load('c2new'), f'TIER 2 (calibrado em {TIER2_CALIB})', 6, len(list(SEEDS_FULL)))

# Controle de reprodutibilidade do braço publicado contra os números já na dissertação.
# ATENÇÃO na leitura: um Δ grande aqui NÃO é necessariamente divergência de versão. Em
# 2026-09-19, ao reproduzir HAB/seed0 localmente, o FT-softmax bateu exatamente (0,6510)
# enquanto o SAINT deu 0,4250 contra 0,5671 — porque a GRADE escolheu outra arquitetura
# (2 cabeças/2 camadas em vez de 4/3). Com o escore de CV praticamente empatado entre as
# seis células, ruído numérico entre placas inverte o argmax. Por isso o diagnóstico
# separa "mesma arquitetura escolhida" de "F1 diferente", e compara o Δ com a dispersão
# entre sementes do próprio dataset.
pub_ref = Path('results/tier1_gridcv.json')
if pub_ref.exists() and _load('c1pub'):
    ref = {(r['variant'], r['dataset'], r['seed']): r
           for r in json.load(open(pub_ref)) if r.get('status') == 'ok'}
    disp = collections.defaultdict(list)
    for r in json.load(open(pub_ref)):
        if r.get('status') == 'ok': disp[(r['variant'], r['dataset'])].append(r['test_f1_macro'])
    ig_par, dif_par, ig_f1 = [], [], 0
    for r in _load('c1pub'):
        k = (r['variant'], r['dataset'], r['seed'])
        if k not in ref: continue
        d = abs(r['test_f1_macro'] - ref[k]['test_f1_macro'])
        mesmo = r.get('best_params') == ref[k].get('best_params')
        (ig_par if mesmo else dif_par).append((d, r['variant'], r['dataset'], r['seed']))
        if mesmo and d < 1e-6: ig_f1 += 1
    print(f'\nreprodutibilidade do braço publicado ({len(ig_par)+len(dif_par)} pares):')
    print(f'  mesma arquitetura escolhida: {len(ig_par)}  (destes, F1 idêntico: {ig_f1})')
    if ig_par:
        pior = max(ig_par)
        print(f'  pior Δ F1 COM mesma arquitetura: {pior[0]:.4f} em {pior[1]}/{pior[2]}/seed{pior[3]}')
        print('    -> este é o número que importa. Muito acima de ~1e-3 = investigar versão.')
    if dif_par:
        print(f'  arquitetura DIFERENTE (empate na grade invertido por ruído): {len(dif_par)}')
        for d, v, ds, sd in sorted(dif_par, reverse=True)[:5]:
            sigma = st.pstdev(disp[(v, ds)]) if len(disp[(v, ds)]) > 1 else float('nan')
            print(f'    {v}/{ds}/seed{sd}: Δ={d:.4f} ({d/sigma:.1f} dp das 30 sementes)')
        print('    -> esperado, e é sintoma do próprio achado: em 40 épocas o escore de CV')
        print('       não separa as arquiteturas, então o argmax é instável.')

if h1 and h2:
    tot = h1 + h2
    print(f'\n>>> execução completa (células 7+8): {tot:.1f} h de ajuste, {tot/2:.1f} h de sessão em 2 placas.')
    if tot/2 > 11:
        print('    ACIMA do limite de 12 h do Kaggle: rode a célula 7 numa sessão e a 8 em outra,')
        print('    ou reduza SEEDS_FULL na célula 4. Os runners são resumíveis (RESUME_DIR).')

In [ ]:
# ── 7. COMPLETO, Tier 1: braço 200/16, 10 datasets × 10 sementes ──
# O braço de 40/6 NÃO é reexecutado: results/tier1_gridcv.json já É esse braço, com a
# seleção de grade dele, pareável nas sementes 0–9. Quem atesta que o código atual
# reproduz aqueles números é o controle de reprodutibilidade da célula 6.
run_phase_2gpu('f1new', "python -u scripts/run_tier1_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER1 + " " + BUDGET +
               " --seeds {seeds} --output {output} --log-level WARNING",
               seeds=SEEDS_FULL)

In [ ]:
# ── 8. COMPLETO, Tier 2: braço 200/16, 6 datasets × 10 sementes ──
run_phase_2gpu('f2new', "python -u scripts/run_tier2_gridcv.py --models " + MODELS_STR +
               " --datasets " + TIER2 + " --n-train 2000 " + BUDGET +
               " --seeds {seeds} --output {output} --log-level WARNING",
               seeds=SEEDS_FULL)
print('\nBaixe de /kaggle/working: budget_full_t1_200.json e budget_full_t2_200.json')